### Understanding how tokenizatin works with first principle
- Converting each sentences into token (smallest possible combination of characters)
- Converting each tokens into token id's
- Making vector positional embedding for each token id's

In [1]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_data = f.read()

print('Character lengths the dataset: ', len(raw_data))

Character lengths the dataset:  20479


In [2]:
import re

# Converting sentences into token with regular expression
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_data)
preprocessed = [item for item in preprocessed if item.strip()]

#print(preprocessed[:20])

In [27]:
#Making a dictionaries of all unique tokens and assigining token id's to them
all_words = set(preprocessed)
# Sort unique words in ascending order
all_words = sorted(all_words)

# Assign id's to each unique token
vocab = {item:integer for integer, item in enumerate(all_words)}
#print(vocab)

### Making encoder and decoder block
- Encoder: - `Text -> Token -> Token id`
- Decoder: - `Token id -> Token -> Text`

In [23]:
class SimpleTokenizationV1:
    def __init__(self, vocab):
        # dictionary to have str to int values
        self.str_to_int = vocab
        # dictionary to have int to str values (decoder)
        self.int_to_str = {v:i for i, v in vocab.items()}

    def encoder(self, text):
        # Converting text into tokens
        processed_text = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        processed_text = [item for item in processed_text if item.strip()]

        # Associating tokens with token id's
        ids = [self.str_to_int[i] for i in processed_text]
        return ids

    def decoder(self, ids):
        # Converting token ids into tokens
        tokens = " ".join([self.int_to_str[i] for i in ids])

        # Remove spaces to match up original text
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', tokens)
        return text

tokenizer = SimpleTokenizationV1(vocab)
text = """
"The height of his glory"--that was what the women called it.
"""
ids = tokenizer.encoder(text)
print(ids)

# Decode back the text produced
print(tokenizer.decoder(ids))

[1, 93, 538, 722, 549, 496, 1, 6, 987, 1077, 1089, 988, 1112, 242, 585, 7]
" The height of his glory" -- that was what the women called it.


In [28]:
# issue when the word or token not there in dictionary
another_token = SimpleTokenizationV1(vocab)
text = """
Hello "The height of his glory"--that was what the women called it.
"""

ids = another_token.encoder(text)


NameError: name 'SimpleTokenizationV1' is not defined

- To mitigate on dealing with unknown tokens in the vocablary,
- Special context tokens are used to better handle the models
- `<|unk|>`: - unknown token used when the token is not present in the vocabalary
- `<|endoftext|>`: - Token used to determine different sources of datasets and it's intersection points

In [70]:
# Adding special tokens in the vocab list
all_words = sorted(list(set(preprocessed)))

# Add special context tokens into vocab dictionary
all_words.extend(["<|unk|>", "<|endoftext|>"])

vocab = {value:item for item, value in enumerate(all_words)}

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|unk|>', 1130)
('<|endoftext|>', 1131)


In [74]:
class SimpleTokenizationV2():
    def __init__(self, vocab):
       self.str_to_int = vocab
       self.int_to_str = {val:idx for idx,val in vocab.items()}

    def encoder(self, text):
        # Convert Text into tokens
        text = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        # Splitting and removing spaces from tokens
        text = [item for item in text if item.strip()]
        # Checking if all tokens are there in dictionary or not
        text = [item if item in self.str_to_int else "<|unk|>" for item in text]
        # Converting token into token id's
        ids = [self.str_to_int[i] for i in text]

        return ids

    def decoder(self, ids):
        # converting id's back to tokens
        text = " ".join([self.int_to_str[i] for i in ids])
        # Remove spaces
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)

        return text

tokenizer = SimpleTokenizationV2(vocab)
text = """
Hello "The height of his glory"--that was what the women called it.
"""
ids = tokenizer.encoder(text)
print(ids)

print(tokenizer.decoder(ids))



[1130, 1, 93, 538, 722, 549, 496, 1, 6, 987, 1077, 1089, 988, 1112, 242, 585, 7]
<|unk|>" The height of his glory" -- that was what the women called it.
